# Issues with Recurrnet Models: 

## Linear Interaction Distance
Nearby words affect each other more easily than distant words. And it takes O(seq_len) for distant words to interact.

## Unparallelizable
Computations across time steps are unparallelizable

# Self Attention to Solve the Issues Above

## Linear Interaction Distance
With self attention, every word in a sentence as a query looks at every word as a key in the sentence.

## Unparallelizable
The looking behaviour is Matrix Multiplication $QK^T$.

# Problems with Self Attention and Ways to Solve them

## Sequence Order
The primitive Self Attention gives the same representation for "Zuko made his uncle" and "his uncle made Zuko". It doesn't encode position information.

### Add Positional Encoding to inputting Embeddings
Types of Positional Encoding: Sinusoidal Position Encodings, Learnable Position Encodings, etc.

#### Sinusoidal Position Encodings
The Sinusoidal Position Encoding of i-th hidden dim of a token is defined as:

$$
PE_{(pos, 2i)}   = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

It has some good properties:

##### Relative Position is Linear Transformation
Let $10000^{2i/d_{model}}$ be a constance $c$, the PE(Position Encoding) of a token at pos, 2i can be denoted as $sin(pos)$, and that at 2i+1 can be denoted as $cos(pos)$. Write this into a column vector:

$$
[sin(pos), cos(pos)]^T
$$

Then the PE at pos+k can be denoted as:

$$
[sin(pos+k), cos(pos+k)]^T = [sin(pos)cos(k) + cos(pos)sin(k), cos(pos)cos(k)-sin(pos)sin(k)]^T
$$

The Transformation from pos to pos+k is

$$
[[cos(k), sin(k)], [-sin(k), cos(k)]]
$$


Pros: Learnable

Cons: Can not extrapolate

#### More Position Representations
Relative linear position attention, Dependency syntax based position representation

### Nonlinearality
By adding Feed Forward Network after the attention layer, nonlinearality is introduced token-wise.

### Sometimes ensure that we don't look at the future: Attention Mask
There are some kind of tokens that we don't want attention to look at, e.g., padding tokens and future tokens in seq2seq decoder. So we mask them. 

We could change the set of keys and values at every time step but it makes computation unparalellizable. So mathematically, we just set the attention scores of masked tokens to negative infinity.

# Transformer
## Masked Multi-Head Self Attention
Intuition: We want multiple attentions to **potentially**(but not really) look at different kinds of information. For example, one attention to look at entities, another to look at syntax.

Let's say we want the ultimate output dim to be d_model, and we have h heads, then each head is of $\frac{d_{model}}{h}$. After we get the outputs of heads, we concat them and give it a linear transformation.

One thing to pay attention, multi-head attention is not necessarily more computationally expensive than single head attention.

<img src="./Mullti-Head_Attention_Computational_Efficiency.jpg" width="400" height="200">

## Scaled Dot Product
The dot product mentioned above is $QK^T$, its variance is subject to model dimension(more specifically, attention head dimension). To cancel out this effect, we can divide it by the square root of the dimension.
$$
\sqrt{d_k}
$$

## Optimization Tricks
Residual Connection and Layer Norm(Add & Norm)

### Layer Norm
For feature dims($H$ features), compute their mean and variance
$$
\mu = \frac{1}{H}\sum^{H}_{i=1}x_i, \sigma^2 = \frac{1}{H}\sum^{H}_{i=1}(x_i - \mu)
$$

Layer Norm output is 

$$
y_i = \gamma_i\frac{(x_i-\mu)}{\sqrt{\sigma^2 + \epsilon}} + \beta_i
$$

, where $\epsilon$ is for numerical stability is usually set to 1e-5.

## Cross Attention
Outputs from the decoder to construct Q, and outputs from the encoder to construct K and V.

## Drawbacks
### Quadratic Computation
A matrix multiplication involves $n^2d$ scalar multiplications, where n is the sequence length and d is the dimension.

In [ ]:
import torch.nn as nn
import torch
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length=1000):
        super().__init__()
        pe = torch.zeros(max_seq_length, d_model)
        pos = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term=torch.exp(torch.arange(0, d_model, 2) * (-torch.log(10000) / d_model))
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        pe = pe.unsqueeze(0) #(batch_size, seq_len, embed_dim)
        torch.register_buffer("pe", pe)

    def forward(self, x):
        length = x.size(1)
        return x + self.pe[:, :length, :]

class SelfAttention(nn.Module):
    